# 20 — Model-Aware Prompt Engineering

## Scenario
We need to deploy a summarization task. 

**The Problem:** We hear rumors that "Model X needs you to ask nicely" or "Model Y needs you to add XML tags." This is **provider folklore**. 

**The Solution:** We design a **Portable Prompt Contract** (a strict Pydantic schema) and evaluate it against multiple models using a unified SDK. We then make a data-driven decision based on Latency, Cost (Tokens), and Correctness, rather than folklore.

In [ ]:
import os
import time
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

client = genai.Client()


## Step 1: The Portable Contract

We define our contract. This schema should work regardless of which model we plug in underneath.

In [ ]:
class EntityExtraction(BaseModel):
    company_name: str = Field(description="The name of the company.")
    founding_year: int = Field(description="The year it was founded.")
    industry: str = Field(description="The industry it operates in.")

prompt = "Extract the entity information from the following text:\n"
text = "In 1998, Google was founded as a search engine company in California."


## Step 2: The Evaluation Function

We build an adapter function that tracks Latency and Token Usage (Cost).

In [ ]:
def evaluate_model(model_name: str, prompt: str, text: str):
    print(f"\n--- Evaluating: {model_name} ---")
    
    start_time = time.time()
    try:
        response = client.models.generate_content(
            model=model_name,
            contents=f"{prompt}\n{text}",
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json",
                response_schema=EntityExtraction,
            )
        )
        latency = time.time() - start_time
        
        # The model should return valid JSON matching our Pydantic schema
        result = EntityExtraction.model_validate_json(response.text)
        
        print(f"Success! Found: {result.company_name} ({result.founding_year}) - {result.industry}")
        print(f"Latency: {latency:.2f} seconds")
        # In the Google GenAI SDK, token counts are available in usage_metadata
        if response.usage_metadata:
             print(f"Tokens Used (Cost): {response.usage_metadata.total_token_count}")
             
    except Exception as e:
        print(f"FAILED! Error: {e}")


## Step 3: Data-Driven Selection

We run the exact same contract against `gemini-2.5-flash` (fast/cheap) and `gemini-2.5-pro` (heavy/smart).

In [ ]:
evaluate_model('gemini-2.5-flash', prompt, text)
evaluate_model('gemini-2.5-pro', prompt, text)


## Conclusion

By evaluating the models objectively using the same contract, you can see if the extra latency and token cost of a "Pro" model is actually justified for your specific task.

If `flash` gets the schema right 100% of the time with half the latency, you choose `flash`. Stop guessing; start evaluating.